# Example: Apple 2020

<h1 align="center" style="margin-bottom: 0;">
    <a href="https://github.com/reeyarn/openesef">
        <img src="https://raw.githubusercontent.com/reeyarn/openesef/refs/heads/master/markdown/esefdata.svg" alt="Open ESEF" style="max-width: 100%; height: auto;"/>
    </a>
</h1>
<h1 align="center" style="margin-top: 0;margin-bottom: 0;">
    A Python Library for ESEF and XBRL Filings
    <br>
        <img src="https://img.shields.io/badge/Project%20Status-Under%20Development-yellow" alt="Project Status: Under Development - 66% Complete" />
        <img src="https://img.shields.io/badge/License-GPLv3-blue.svg" alt="License: GPL v3.0" />
</h1>
<h1 align="center" style="margin-top: 0;margin-bottom: 0;">
<div align="center" style="font-size: 14px; margin-top: 0; margin-bottom: 0;">
<a href="https://github.com/reeyarn/openesef">github.com/reeyarn/openesef</a>
</div>
</h1>

First, lets load the logger:

In [1]:
import logging
from openesef.util.util_mylogger import setup_logger 
logger = setup_logger("main", logging.CRITICAL, log_dir="/tmp/log/")

Now, lets load the xbrl filing, using Apple 2020 as an example:

In [2]:
from openesef.edgar.loader import load_xbrl_filing
xid, tax = load_xbrl_filing(ticker="AAPL", year=2020)
#xid, tax = load_xbrl_filing(filing_url="/Archives/edgar/data/320193/0000320193-20-000096.txt")

The xbrl filing is loaded and the taxonomy is created.

Now, lets print the XBRL instance (xid):

In [3]:
print(xid)


Namespaces: 10
Schema references: 1
Linkbase references: 0
Contexts: 318
Units: 9
Facts: 1388
Footnotes: 0
Filing Indicators: 0


And the taxonomy (tax):

In [4]:
print(tax)

Schemas: 14
Linkbases: 7
Role Types: 775
Arcrole Types: 0
Concepts: 18293
Item Types: 52
Tuple Types: 0
Simple Types: 0
Labels: 0
References: 0
Hierarchies: 79
Dimensional Relationship Sets: 79
Dimensions: 295
Hypercubes: 363
Enumerations: 0
Enumerations Sets: 0
Table Groups: 0
Tables: 0
Parameters: 0
Assertion Sets: 0
Value Assertions: 0
Existence Assertions: 0
Consistency Assertions: 0


DEI stands for Document and Entity Information. For each XBRL report, there will be a section for DEI,
and this class is to provide easy access to those commonly-defined DEI attributes.

Lets print the DEI:

In [5]:
for i, (key, value) in enumerate(xid.dei.items()):
    print(f"{i}: {key}: {value}")
    if i>7:
        break


0: AmendmentFlag: false
1: DocumentFiscalYearFocus: 2020
2: DocumentFiscalPeriodFocus: FY
3: EntityCentralIndexKey: 0000320193
4: CurrentFiscalYearEndDate: --09-26
5: DocumentType: 10-K
6: DocumentAnnualReport: true
7: DocumentPeriodEndDate: 2020-09-26
8: DocumentTransitionReport: false


In [6]:
from openesef.engines.tax_pres import TaxonomyPresentation
t_pres = TaxonomyPresentation(tax)


In [7]:
print("\nConcept Labels in Statement of Operations:")
concepts_statement_of_operations = []

iprint=0
for concept in t_pres.statement_concepts.values():
    if concept['statement_name'] == 'CONSOLIDATEDSTATEMENTSOFOPERATIONS':
        concepts_statement_of_operations.append(concept['concept_qname'])
        print("-"*30)
        print(f"Statement: {concept['statement_name']}")
        print(f"Concept: {concept['concept_qname']}")
        print(f"Label: {concept['label']}")        
        iprint+=1
        if iprint>3:
            break
    



Concept Labels in Statement of Operations:
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: us-gaap:IncomeStatementAbstract
Label: Income Statement [Abstract]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: srt:ProductOrServiceAxis
Label: Product and Service [Axis]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: srt:ProductsAndServicesDomain
Label: Product and Service [Domain]
------------------------------
Statement: CONSOLIDATEDSTATEMENTSOFOPERATIONS
Concept: us-gaap:ProductMember
Label: Products


In [8]:
# Get the current year's main instance context
periods_dict = xid.identify_reporting_contexts()
#import pandas as pd
#print(pd.DataFrame.from_dict(periods_dict, orient='index'))


In [9]:
current_contexts = [ctx_id for ctx_id, ctx_info in periods_dict.items() 
                    if ctx_info['relative_year'] == 0 and  ctx_info.get('main_context')]

print(current_contexts)

['i747bec89b4e84f74ae3445db3509f609_I20200926', 'i223bd574caab4f739f73936be6065c72_D20190929-20200926', 'ic3ea678a3e394e00880d68882e8bdc02_I20200327', 'i5085fb79a9a14a9aae9b909beb32bce2_D20180930-20190928', 'i4920908218084be688c8fa96b8903033_D20171001-20180929']


In [10]:
print("\nFact Values:")
iprint=0
for key, fact in xid.xbrl.facts.items():
    concept_qname = fact.qname if hasattr(fact, 'qname') else 'N/A'  # Get the concept's QName
    context = xid.xbrl.contexts[fact.context_ref]
    period_info = periods_dict.get(fact.context_ref, {})
    period_string = period_info.get('period_string', 'N/A')
    if concept_qname in concepts_statement_of_operations and fact.context_ref in current_contexts:
        print(f"{concept_qname:<90} Value: {fact.value:<15} Context: {period_string}")    
        iprint+=1
        if iprint>7:
            break


Fact Values:


In [11]:
def is_numeric(x):
    try:
        float(x)
        return True
    except (ValueError, TypeError):
        return False


In [12]:
from openesef.engines.tax_pres import ins_facts

fact_df = ins_facts(xid, tax)
fact_df["val_mln"] = fact_df["value"].apply(lambda x: float(x)/1000000 if is_numeric(x) and float(x) > 1000000 else x)
fact_df.sort_values(by='fact_index', inplace=True)
fact_df = fact_df.loc[fact_df.fact_included ]
current_period_string = fact_df.period_string.value_counts().index[0]
current_facts = fact_df[fact_df.period_string == current_period_string].reset_index(drop=True)

current_facts.loc[(current_facts['statement_name'] == 'CONSOLIDATEDSTATEMENTSOFOPERATIONS')  , ['fact_index', 'concept_name', 'label', "segment_axis", 'val_mln', 'period_end']].head(30)

/Users/mbp16/Dropbox/sciebo/WebScraping+ESEF_Paper/Research/code_fse/openesef_repo/openesef/engines/tax_pres.py:1228: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fact_df["fact_included"] = fact_df["fact_included"].fillna(False)


,fact_index,concept_name,label,segment_axis,val_mln,period_end
0,82,RevenueFromContractWithCustomerExcludingAssess...,Products,srt:ProductOrServiceAxis,220747.0,2020-09-26
1,85,RevenueFromContractWithCustomerExcludingAssess...,Services,srt:ProductOrServiceAxis,53768.0,2020-09-26
2,88,RevenueFromContractWithCustomerExcludingAssess...,Net sales,None,274515.0,2020-09-26
3,91,CostOfGoodsAndServicesSold,Products,srt:ProductOrServiceAxis,151286.0,2020-09-26
4,94,CostOfGoodsAndServicesSold,Services,srt:ProductOrServiceAxis,18273.0,2020-09-26
5,97,CostOfGoodsAndServicesSold,Cost of sales,None,169559.0,2020-09-26
6,100,GrossProfit,Gross margin,None,104956.0,2020-09-26
7,103,ResearchAndDevelopmentExpense,Research and development,None,18752.0,2020-09-26
8,106,SellingGeneralAndAdministrativeExpense,"Selling, general and administrative",None,19916.0,2020-09-26
9,109,OperatingExpenses,Operating Expenses,None,38668.0,2020-09-26


The above is the Statement of Operations from XBRL extraction; below is a screenshot of the same statement from the Form 10-K filing.

<img src="https://raw.githubusercontent.com/reeyarn/openesef/refs/heads/master/examples/apple20200926.png" width="600px">

Below is the income statement R2.htm from the Form 10-K filing.

In [13]:
#https://www.sec.gov/Archives/edgar/data/320193/000032019320000096/R2.htm

import requests
from IPython.display import HTML

# Fetch the content
response = requests.get('https://www.sec.gov/Archives/edgar/data/320193/000032019320000096/R2.htm', 
                    headers={"user-agent": "Your Name youremail@your.domain.com"})
html_content = response.text

# Display the HTML content
HTML(html_content)